# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All references are by their `@id`.

We'll print the available record sets and for each, the fields and their `@id`s.

In [ ]:
# List all available record sets by their @id
print('Available record sets:')
record_set_ids = [rs['@id'] for rs in metadata.to_json().get('recordSet', [])]
for i, rs_id in enumerate(record_set_ids):
    print(f"  {i+1}. {rs_id}")

# For each record set, print its fields and their @id
for rs in metadata.to_json().get('recordSet', []):
    print(f"\nRecord Set @id: {rs['@id']}")
    fields = rs.get('field', [])
    if not isinstance(fields, list): fields = [fields]
    print('  Fields:')
    for f in fields:
        if isinstance(f, dict):
            print(f"    - {f.get('@id')}: {f.get('name')}")
        else:
            print(f"    - {f}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set (@id)

# For this dataset, the main data table is typically named with a descriptive @id. We'll extract all record sets listed.
record_sets = [rs['@id'] for rs in metadata.to_json().get('recordSet', [])]
dataframes = {}

for record_set_id in record_sets:
    print(f"Loading records for {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  -> {len(df)} records, columns: {df.columns.tolist()}")
    if len(df) > 0:
        display(df.head(2))

# For rest of the notebook, select main record set if present (e.g., protein data)
# If no record sets present, the notebook will skip further steps.
if len(record_sets) > 0:
    main_record_set_id = record_sets[0]
    main_df = dataframes[main_record_set_id]
    print(f"\nMain dataframe columns for {main_record_set_id}: {main_df.columns.tolist()}")
else:
    print('No record sets available in this metadata.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming distributions, or grouping data by key attributes.

In [ ]:
# Proceed only if data is available
if len(record_sets) == 0 or main_df.shape[0] == 0:
    print('No records to analyze.')
else:
    # Select a numeric field for demonstration.
    # We'll heuristically look for a column with 'MW', 'Abundance', 'Count', or numeric-like name.
    from numpy import number
    import re
    candidates = [col for col in main_df.columns if re.search(r'(MW|abundance|count|coverage|peptide|score|mass|amount|percentage|total|intensity|normalized|value|quantity)', col, re.I)]
    print(f"Candidate numeric fields: {candidates}")
    
    # Let's use the first, try to coerce to numeric if needed
    if len(candidates) == 0:
        print('No obvious numeric field found.')
    else:
        numeric_field = candidates[0]
        # Coerce to numeric
        main_df[numeric_field] = pd.to_numeric(main_df[numeric_field], errors='coerce')
        
        threshold = main_df[numeric_field].dropna().median()
        # Filter for values above median
        filtered_df = main_df[main_df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.3f} (median): ")
        display(filtered_df.head())

        # Normalize the selected field
        mean = filtered_df[numeric_field].mean()
        std = filtered_df[numeric_field].std()
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - mean) / std
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by a string/categorical field (column with <20 unique values, non-numeric, not same as numeric_field)
        categorical_fields = [col for col in main_df.columns if main_df[col].nunique() < 20 and col != numeric_field and not pd.api.types.is_numeric_dtype(main_df[col])]
        if len(categorical_fields) > 0:
            group_field = categorical_fields[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field} (mean {numeric_field}):")
            display(grouped_df.head())
        else:
            print('No suitable categorical field found for grouping.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(record_sets) > 0 and main_df.shape[0] > 0 and len(candidates) > 0:
    # Distribution plot
    plt.figure(figsize=(8,5))
    sns.histplot(main_df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If we identified a group field, bar plot group means
    if len(categorical_fields) > 0:
        plt.figure(figsize=(10,4))
        sns.barplot(x=grouped_df[group_field], y=grouped_df[numeric_field])
        plt.xticks(rotation=45)
        plt.title(f'Mean {numeric_field} by {group_field}')
        plt.show()
else:
    print('Not enough data for visualization.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

*In this notebook, we demonstrated how to use the `mlcroissant` library to load and explore the FAIR^2 dataset on mass spectrometry analysis of extracellular vesicles. We reviewed the available record sets and fields by their `@id`, loaded them into pandas DataFrames, and performed exploratory analysis including filtering and normalizing a numeric field. Simple visualizations revealed data distributions and allowed for grouping by categorical attributes. The `mlcroissant` library facilitates programmatic and reproducible FAIR data workflows.*